# Kombinierte RBF-SVM: Index-Katalog

**Hypothese:** Der Index-Katalog einschließlich `REIP_LAGRANGIAN` und `REIP_POLYNOMIAL` sowie die optionale PCA-Merkmalsgruppe `PC_1`/`PC_2` verbessern die kombinierte Balanced Accuracy der RBF-SVM gegenüber Rohspektren.

Die vier Varianten unterscheiden sich ausschließlich in diesen Merkmalsgruppen. Alle verwenden dieselben Datenartefakte, die numerische Monatskodierung, eine Standardisierung der Rohspektren und dasselbe Raster für `C` und `gamma`. Die Auswahl erfolgt ausschließlich anhand der Cross-Validation auf dem Trainingsanteil; nur der CV-Gewinner wird einmal auf dem gemeinsamen Validierungsanteil bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

EXPERIMENT_ID = "combined_svm_index_reip_pca"
TRACKING_EXPERIMENT = "vegetation-index-features"
PARAM_GRID = {
    "estimator__C": [30.0, 100.0, 300.0],
    "estimator__gamma": [0.001, 0.003, 0.01],
}
BASE_PREPROCESSING = {
    "use_meta": True,
    "use_cyclic_month": False,
    "use_spectral_standard_scale": True,
}
FEATURE_VARIANTS = {
    "Rohspektren": PreprocessingConfig(**BASE_PREPROCESSING),
    "Index-Katalog inkl. REIP": PreprocessingConfig(
        **BASE_PREPROCESSING,
        use_vegetation_indices=True,
    ),
    "PC_1 + PC_2": PreprocessingConfig(
        **BASE_PREPROCESSING,
        use_principal_components=True,
    ),
    "Index-Katalog inkl. REIP + PC_1 + PC_2": PreprocessingConfig(
        **BASE_PREPROCESSING,
        use_vegetation_indices=True,
        use_principal_components=True,
    ),
}
VARIANT_SLUGS = {
    "Rohspektren": "raw",
    "Index-Katalog inkl. REIP": "indices_reip",
    "PC_1 + PC_2": "principal_components",
    "Index-Katalog inkl. REIP + PC_1 + PC_2": "indices_reip_principal_components",
}

## 1. Varianten und Hyperparameter auf Trainings-Folds suchen

Für jede der vier Feature-Varianten werden dieselben neun Kombinationen aus `C` und `gamma` ausschließlich auf den gemeinsamen Trainings-Folds nach kombinierter Balanced Accuracy bewertet. Die Variantenwahl erfolgt allein über diesen CV-Score; der Validierungsanteil bleibt bis zur nächsten Zelle unberührt.

In [ ]:
tuned_by_variant = {}
for variant_name, preprocessing in FEATURE_VARIANTS.items():
    variant_slug = VARIANT_SLUGS[variant_name]
    tuned_by_variant[variant_name] = tune(
        make_combined_svm(),
        TuneConfig(
            name=f"{EXPERIMENT_ID}_{variant_slug}_search",
            approach="combined",
            experiment=TRACKING_EXPERIMENT,
            description=(
                f"RBF-SVM mit {variant_name}; C und gamma werden bei ansonsten gleicher "
                "Vorverarbeitung per CV verglichen."
            ),
            preprocessing=preprocessing,
            param_grid=PARAM_GRID,
        ),
    )

tuning_summary = (
    pd.DataFrame(
        [
            {
                "Variante": variant_name,
                "CV-BAcc kombiniert": tuned.best_score,
                "Hyperparameter": tuned.best_params,
                "Tuning-Run": tuned.run_id,
            }
            for variant_name, tuned in tuned_by_variant.items()
        ]
    )
    .sort_values("CV-BAcc kombiniert", ascending=False)
    .reset_index(drop=True)
)
winner_name = tuning_summary.loc[0, "Variante"]
tuned = tuned_by_variant[winner_name]

display(tuning_summary)
{"CV-Gewinner": winner_name, "best_params": tuned.best_params, "tuning_run": tuned.run_id}

## 2. CV-Gewinner einmal auf der Validierung bewerten

Diese Zelle erst ausführen, nachdem die CV-Rangfolge in der vorherigen Zelle feststeht. Sie bewertet ausschließlich die beste Variante mit ihren per CV ausgewählten Hyperparametern einmal auf dem gemeinsamen Validierungsanteil.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name=f"{EXPERIMENT_ID}_{VARIANT_SLUGS[winner_name]}",
        approach="combined",
        experiment=TRACKING_EXPERIMENT,
        description=(
            f"CV-Gewinner ({winner_name}) des kontrollierten Vergleichs von Index-Katalog, "
            "REIP und PCA-Komponenten."
        ),
        preprocessing=FEATURE_VARIANTS[winner_name],
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. CV-Rangfolge und Holdout-Ergebnis einordnen

Die Tabelle vergleicht alle Varianten ausschließlich über ihre CV-BAcc. Der Validierungs-Score erscheint nur für den vorab ausgewählten CV-Gewinner. Die Diagramme zeigen dessen Hyperparameterstreuung, Fehlerstruktur und häufigste Verwechslungen für Kultur und Stadium.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = tuning_summary.copy()
comparison["Validierungs-BAcc kombiniert"] = pd.NA
comparison.loc[
    comparison["Variante"] == winner_name,
    "Validierungs-BAcc kombiniert",
] = result.metrics.bacc_combined
display(comparison)
display(result.metrics.model_dump())

top_confusions = (
    pd.DataFrame(
        {
            "Wahr": combined_label(result.y_val),
            "Vorhergesagt": combined_label(result.y_pred),
        }
    )
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
display(top_confusions)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title(f"Kombinierte RBF-SVM: {winner_name}")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

## 4. Abbildungen für die Dokumentation sichern

Diese Zelle nur ausführen, wenn der CV-Gewinner im Experiment-Log und auf der Ansatzseite dokumentiert wird.

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(
    tuning_ax.figure,
    f"{EXPERIMENT_ID}_tuning",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_confusion_matrices(result.y_val, result.y_pred),
    f"{EXPERIMENT_ID}_confusion",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_class_recall(result.y_val, result.y_pred),
    f"{EXPERIMENT_ID}_recall",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_top_confusions(result.y_val, result.y_pred),
    f"{EXPERIMENT_ID}_confusions",
    MODEL_DOCS_FIGURES_DIR,
)
{
    "cv_winner": winner_name,
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "invalid_combinations": result.metrics.invalid_combinations,
}